# 06 · Exportar los datos para Power BI
Exporta a CSV las tablas limpias y los resultados del análisis, en la carpeta `powerbi/datos`.
Antes hay que haber ejecutado `04_limpieza` y `05_analisis`.

Se usa CSV porque Power BI lo lee directamente, sin tener que instalar nada para conectar con SQLite.
Los ficheros se guardan con punto decimal y fechas aaaa-mm-dd: al cargarlos en Power BI hay que
elegir la configuración regional **Inglés (Estados Unidos)** para que lea bien los decimales.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

PROYECTO = Path(r"C:\Users\Annag\Desktop\portfolio\caso1_reformas_alzina_datos")
bd = sqlite3.connect(PROYECTO / "sql" / "alzina.db")
carpeta = PROYECTO / "powerbi" / "datos"
carpeta.mkdir(parents=True, exist_ok=True)

tablas = {
    # Dimensiones (los "quién" y "qué")
    "dim_clientes": "SELECT * FROM limpio_clientes",
    "dim_obras": """SELECT o.codigo AS obra, o.cliente_id, o.poblacion, o.tipo, o.fecha_inicio, o.fecha_fin,
                           o.presupuesto, COALESCE(o.observaciones, '') AS observaciones
                    FROM limpio_obras AS o""",
    # Hechos (los "cuánto" y "cuándo")
    "hechos_facturas": """SELECT f.num_factura, f.fecha, f.vencimiento, f.cliente_id, f.obra, f.base, f.iva_pct,
                                 f.cuota_iva, f.total, COALESCE(f.rectifica_a, '') AS rectifica_a
                          FROM limpio_facturas AS f""",
    "hechos_estado_facturas": "SELECT * FROM estado_facturas",
    "hechos_cobros": """SELECT cf.fila_banco, c.fecha, c.cliente_id, cf.num_factura, cf.importe, cf.metodo,
                               f.fecha AS fecha_factura,
                               JULIANDAY(c.fecha) - JULIANDAY(f.fecha) AS dias_hasta_cobro,
                               JULIANDAY(c.fecha) - JULIANDAY(f.vencimiento) AS dias_de_retraso
                        FROM limpio_cobros_facturas AS cf
                        JOIN limpio_cobros AS c USING (fila_banco)
                        LEFT JOIN limpio_facturas AS f ON f.num_factura = cf.num_factura""",
    "hechos_gastos": "SELECT * FROM limpio_gastos",
    "hechos_horas": "SELECT * FROM limpio_partes",
    "hechos_nominas": "SELECT * FROM limpio_nominas",
    # Resultados del análisis
    "analisis_margen_obras": "SELECT * FROM analisis_margen_obras",
    "analisis_prevision_caja": "SELECT * FROM analisis_prevision_caja",
    "analisis_saldo_diario": "SELECT * FROM analisis_saldo_diario",
}

resumen = []
for nombre, consulta in tablas.items():
    df = pd.read_sql(consulta, bd)
    df.to_csv(carpeta / f"{nombre}.csv", index=False, encoding="utf-8-sig")
    resumen.append((nombre, len(df), len(df.columns)))
pd.DataFrame(resumen, columns=["fichero", "filas", "columnas"])